# riverWQ EDA — Predicting E. coli Contamination in NZ Rivers

## Project Introduction

This notebook covers the exploratory data analysis required before the final modelling begins. The findings and conclusions will be used for preprocessing and modelling in '02_modelling.ipyb'.

# 1. Setup & Data Loading

## 1.1 Library Import

I will import the following libraries for EDA.

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Encoding tool
from sklearn.preprocessing import LabelEncoder
# Scaler tool
from sklearn.preprocessing import StandardScaler
# Data split tool
from sklearn.model_selection import train_test_split
# LogisticRegression method tool
from sklearn.linear_model import LogisticRegression
# KNN model method tool
from sklearn.neighbors import KNeighborsClassifier
# Random Forest method tool
from sklearn.ensemble import RandomForestClassifier
# Classification report tool
from sklearn.metrics import classification_report
# Confusion metrix tool
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

## 1.2 Dataset Loading 

In the following code cell, main dataset will be loaded.

In [ ]:
# Load the main dataset
df = pd.read_csv('../data/riverWQ_dataset.csv')


# 2. Initial Profiling

In this section, basic knowledge of the dataset will be worked out. And the quality and potential problems of the dataset will be monitored.

## 2.1 Dataset preview

In the following code cell, the basic information and overlook of dataset will be applied.

In [ ]:
# Show the shape of the dataset
print(f'Dataset shape: {df.shape}')
# Show the first 5 rows
df.head().T

## 2.2 Data types 

In this section, data types of the dataset will be shown to monitor the potential mismatch problems.

In [ ]:
# Show the dtype of each column
df.dtypes

**Data type observations**

All dtypes are stored properly, and no conversions are needed at this stage.

## 2.3 Missing value observation

In this section, missing values will be observed in the dataset.

In [30]:
df.isnull().sum()

site_id                                          0
region                                           0
catchment                                        0
latitude                                         0
longitude                                        0
wfs_land_cover                                   0
wfs_altitude                                     0
rec_land_cover_type                              0
mean_annual_rainfall_mm                          0
pct_cropland                                     0
pct_forest                                       0
pct_grassland_other_herbaceous_vegetation        0
pct_scrub_shrubland                              0
pct_urban_bare_lightly-vegetated_surfaces        0
pct_cropping_horticulture                        0
pct_exotic_forest                                0
pct_exotic_grassland                             0
pct_exotic_scrub_shrubland                       0
pct_indigenous_forest                            0
pct_indigenous_scrub_shrubland 

**Missing value findings**

13 columns have missing values and most of them show up in pairs('band' and 'median') apart from 'nitrate_band':

- 1. Target variables:
'ecolin_median' and 'ecoli_band' are both target variables, and each of them have 57 missing values. These rows can't be used for training or evaluation. Since they are target variables, further actions will be needed and discussed.

- 2. Other columns:
Other columns that have missing values such as 'no3n_band' and so on represent water quality outcomes measured alongside E. coli, so they will be excluded later from modelling to avoid data leakage. Thus, no more actions are needed.

## 2.4 Duplicates

In this section, I will check whether the duplicates exist in the dataset.

In [29]:
# Count exact duplicate rows
df.duplicated().sum()

np.int64(0)

**Observations**

There are no duplicates being observed in the dataset.

## 2.5 Implausible values

In this section, numerical values will be checked through to identify any implausible values or unusual values.

In [34]:
# Summary statistics for all numerical columns
df.describe().T

,count,mean,std,min,25%,50%,75%,max
latitude,396.0,-40.749636,3.028036,-46.515137,-43.205942,-39.794082,-38.400809,-36.344941
longitude,396.0,173.810139,2.858883,167.686111,170.999559,174.995460,175.771552,178.427553
mean_annual_rainfall_mm,396.0,1486.785354,596.707801,542.000000,1092.000000,1413.000000,1677.250000,3373.000000
pct_cropland,396.0,1.475682,4.309309,0.000000,0.060000,0.520000,0.990000,34.740000
pct_forest,396.0,30.286111,20.895960,0.000000,14.040000,32.680000,41.012500,87.260000
pct_grassland_other_herbaceous_vegetation,396.0,53.139470,23.823439,0.000000,36.140000,54.570000,77.900000,93.550000
pct_scrub_shrubland,396.0,6.752121,6.250243,0.000000,3.600000,4.830000,8.490000,52.560000
pct_urban_bare_lightly-vegetated_surfaces,396.0,6.395101,14.950721,0.000000,1.260000,2.290000,3.890000,100.000000
pct_cropping_horticulture,396.0,1.475682,4.309309,0.000000,0.060000,0.520000,0.990000,34.740000
pct_exotic_forest,396.0,9.260631,7.952776,0.000000,3.062500,7.345000,14.700000,44.330000


In [36]:
# Verify the two columns are truly identical
(df['pct_cropland'] == df['pct_cropping_horticulture']).all()

np.True_

**Observations**

No implausible or impossible values are identified across all attributes in the dataset.

While 'pct_cropland' and 'pct_cropping_horticulture' have identical summary statistics (same mean, std, min, max). And verification confirms that they contain identical values. This finding needs discussing furtherly later on.

# Univariate Analysis